# Atividade: Agentes clássicos — **Blackjack-v1** (Gymnasium)

**Aluno(a):** *Samara A. Almeida*  

Este notebook segue o código base dos materiais de **Q-Learning** e **SARSA** (aula), adaptado para estados **tupla** com tabela `Q` em `defaultdict`. O ambiente é tabular: soma do jogador, carta visível do dealer e indicador de ás utilizável.

### Por que Blackjack-v1?

O espaço de estados é **compacto e discreto**, adequado a métodos tabulares; existe literatura com política ótima de referência (Sutton & Barto), o que facilita interpretar se o agente aprendeu algo sensato; e há apenas **duas ações** (pedir carta ou parar), o que simplifica treino e comparação entre algoritmos.

## Dependências

`pygame` é necessário para `render_mode="rgb_array"` no Blackjack (Toy Text).

In [ ]:
%pip install gymnasium numpy matplotlib tqdm pygame -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import random
from collections import defaultdict
from tqdm.auto import trange

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

ENV_ID = "Blackjack-v1"
N_ACTIONS = 2  # 0 = stick (parar), 1 = hit (pedir)

---
## Tarefa 1 — Exploração do ambiente (0,5 pt)

### Espaço de estados, ações e recompensas

- **Observação:** tupla `(player_sum, dealer_card, usable_ace)` com tipos discretos compatíveis com o ambiente Gymnasium.
- **Ações:** `0` parar (stick), `1` pedir carta (hit).
- **Recompensa:** ao terminar o episódio: `+1` vitória, `0` empate, `-1` derrota. Recompensas intermediárias são `0`.

In [ ]:
env_demo = gym.make(ENV_ID, render_mode="rgb_array")
print("Observation space:", env_demo.observation_space)
print("Action space:", env_demo.action_space)

state, info = env_demo.reset(seed=SEED)
print("Exemplo de estado inicial:", state)

frame = env_demo.render()
plt.figure(figsize=(6, 5))
plt.imshow(frame)
plt.axis("off")
plt.title("Visualização do ambiente (render)")
plt.show()

### Agente aleatório — 100 episódios

Métricas: **taxa de vitória** (`reward == 1`), **recompensa média** por episódio e **número médio de ações** (passos) até o fim do episódio.

In [ ]:
def run_random_episodes(env, n_episodes=100, base_seed=0):
    wins = 0
    rewards = []
    steps_list = []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=base_seed + ep)
        done = False
        steps = 0
        ep_reward = 0.0
        while not done:
            a = env.action_space.sample()
            s, r, term, trunc, _ = env.step(a)
            ep_reward += r
            steps += 1
            done = term or trunc
        wins += int(ep_reward > 0)
        rewards.append(ep_reward)
        steps_list.append(steps)
    return {
        "taxa_vitoria": wins / n_episodes,
        "recompensa_media": float(np.mean(rewards)),
        "passos_medios": float(np.mean(steps_list)),
    }


env_rand = gym.make(ENV_ID)
stats = run_random_episodes(env_rand, n_episodes=100, base_seed=100)
env_rand.close()
print(stats)

---
## Tarefa 2 — Expected SARSA (1 pt)

### Referência e equação

**Expected SARSA** atualiza com o valor esperado da próxima ação sob a política **ε-greedy** atual, em vez da ação amostrada (como no SARSA on-policy clássico):

$$Q(s,a) \leftarrow Q(s,a) + \alpha \big[ R + \gamma \sum_{a'} \pi(a'\mid s')\, Q(s',a') - Q(s,a) \big]$$

Para política **ε-greedy** com $|A|$ ações, pode-se usar a forma equivalente (ver *Reinforcement Learning: An Introduction*, Sutton & Barto, 2ª ed., seção sobre métodos TD / Expected Sarsa):

$$\sum_{a'} \pi(a'\mid s') Q(s',a') = \frac{\varepsilon}{|A|} \sum_{a'} Q(s',a') + (1-\varepsilon) \max_{a'} Q(s',a')$$

(Esta forma assume que a parte *greedy* da política coloca toda a massa $(1-\varepsilon)$ nas ações ótimas de maior valor; em empates de máximo, implementações mais refinadas repartem essa massa entre todas as ações máximas — aqui usamos a expressão acima, como é comum em exercícios tabulares.)

**Fonte:** Richard S. Sutton e Andrew G. Barto, *Reinforcement Learning: An Introduction*, 2nd edition, MIT Press, 2018 (capítulo de métodos de diferença temporal).

### Diferença conceitual (3–5 linhas)

O **SARSA** é *on-policy* e usa o par $(s', a')$ onde $a'$ é a ação **realmente** escolhida no próximo estado, incluindo exploração acidental. O **Expected SARSA** usa o **valor esperado** sob a política atual no próximo estado, o que **suaviza a variância** da atualização porque não depende de um único sorteio de $a'$. Na prática, isso costuma deixar o aprendizado mais estável quando há muita exploração, aproximando-se do comportamento esperado da política ε-greedy em vez de um único trajeto amostrado.

In [ ]:
def new_q_table():
    return defaultdict(lambda: np.zeros(N_ACTIONS, dtype=np.float64))


def epsilon_greedy_policy(Q, state, epsilon, rng):
    if rng.uniform(0, 1) > epsilon:
        return int(np.argmax(Q[state]))
    return rng.integers(0, N_ACTIONS)


def expected_next_q(Q, state, epsilon):
    q = Q[state]
    return (epsilon / N_ACTIONS) * np.sum(q) + (1.0 - epsilon) * np.max(q)


def train_expected_sarsa(
    env,
    Q,
    n_episodes,
    learning_rate,
    gamma,
    min_epsilon,
    max_epsilon,
    decay_rate,
    rng,
):
    rewards_hist = []
    for episode in range(n_episodes):
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
        state, _ = env.reset()
        done = False
        ep_reward = 0.0
        while not done:
            action = epsilon_greedy_policy(Q, state, epsilon, rng)
            new_state, reward, term, trunc, _ = env.step(action)
            done = term or trunc
            ep_reward += reward
            next_ev = 0.0 if done else expected_next_q(Q, new_state, epsilon)
            td_target = reward + gamma * next_ev
            Q[state][action] += learning_rate * (td_target - Q[state][action])
            state = new_state
        rewards_hist.append(ep_reward)
    return Q, np.array(rewards_hist, dtype=np.float64)


---
## Funções de treino e avaliação — Q-Learning e SARSA

(Estrutura análoga aos notebooks de aula; estados são **tuplas** indexando `defaultdict`.)

In [ ]:
def train_q_learning(
    env,
    Q,
    n_episodes,
    learning_rate,
    gamma,
    min_epsilon,
    max_epsilon,
    decay_rate,
    rng,
):
    rewards_hist = []
    for episode in range(n_episodes):
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
        state, _ = env.reset()
        done = False
        ep_reward = 0.0
        while not done:
            action = epsilon_greedy_policy(Q, state, epsilon, rng)
            new_state, reward, term, trunc, _ = env.step(action)
            done = term or trunc
            ep_reward += reward
            best_next = 0.0 if done else float(np.max(Q[new_state]))
            Q[state][action] += learning_rate * (
                reward + gamma * best_next - Q[state][action]
            )
            state = new_state
        rewards_hist.append(ep_reward)
    return Q, np.array(rewards_hist, dtype=np.float64)


def train_sarsa(
    env,
    Q,
    n_episodes,
    learning_rate,
    gamma,
    min_epsilon,
    max_epsilon,
    decay_rate,
    rng,
):
    rewards_hist = []
    for episode in range(n_episodes):
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
        state, _ = env.reset()
        action = epsilon_greedy_policy(Q, state, epsilon, rng)
        done = False
        ep_reward = 0.0
        while not done:
            new_state, reward, term, trunc, _ = env.step(action)
            done = term or trunc
            ep_reward += reward
            next_action = (
                0
                if done
                else epsilon_greedy_policy(Q, new_state, epsilon, rng)
            )
            next_q = 0.0 if done else float(Q[new_state][next_action])
            Q[state][action] += learning_rate * (
                reward + gamma * next_q - Q[state][action]
            )
            if done:
                break
            state = new_state
            action = next_action
        rewards_hist.append(ep_reward)
    return Q, np.array(rewards_hist, dtype=np.float64)


def evaluate_greedy(env, Q, n_episodes, seed_offset=0):
    rewards = []
    for ep in range(n_episodes):
        s, _ = env.reset(seed=seed_offset + ep)
        done = False
        total = 0.0
        while not done:
            a = int(np.argmax(Q[s]))
            s, r, term, trunc, _ = env.step(a)
            total += r
            done = term or trunc
        rewards.append(total)
    r = np.array(rewards, dtype=np.float64)
    return float(r.mean()), float(r.std(ddof=1)) if n_episodes > 1 else 0.0


def smooth_curve(x, window=100):
    if window <= 1:
        return x
    kernel = np.ones(window) / window
    return np.convolve(x, kernel, mode="valid")


---
## Tarefa 3 — Experimento de hiperparâmetros com **Q-Learning** (1,0 pt)

Variamos **α** (`learning_rate`) e **γ** (`gamma`), mantendo os demais parâmetros fixos. Para cada hiperparâmetro usamos **3 valores**. Curvas com **média móvel** (janela configurável).

*Ajuste `N_EP_TRAIN` se quiser treinos mais longos (melhor convergência, mais tempo de CPU).*

In [ ]:
# Aumente (ex.: 300k–500k) para curvas mais estáveis; comece com menos para testar.
N_EP_TRAIN = 100_000
SMOOTH_WIN = max(500, N_EP_TRAIN // 50)

fixed_gamma = 1.0
fixed_alpha = 0.05
min_epsilon = 0.05
max_epsilon = 1.0
decay_rate = 3e-5

alphas = [0.01, 0.05, 0.15]
gammas = [0.95, 0.99, 1.0]

rng_q = np.random.default_rng(SEED)

curves_alpha = {}
for a in alphas:
    env = gym.make(ENV_ID)
    Q = new_q_table()
    _, rh = train_q_learning(
        env,
        Q,
        N_EP_TRAIN,
        learning_rate=a,
        gamma=fixed_gamma,
        min_epsilon=min_epsilon,
        max_epsilon=max_epsilon,
        decay_rate=decay_rate,
        rng=rng_q,
    )
    env.close()
    curves_alpha[a] = rh

plt.figure(figsize=(10, 5))
for a in alphas:
    y = smooth_curve(curves_alpha[a], SMOOTH_WIN)
    plt.plot(y, label=f"α={a}")
plt.xlabel("Episódio (após suavização, eixo deslocado)")
plt.ylabel("Recompensa média (janela móvel)")
plt.title(f"Q-Learning: variação de α (γ={fixed_gamma} fixo)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

curves_gamma = {}
for g in gammas:
    env = gym.make(ENV_ID)
    Q = new_q_table()
    _, rh = train_q_learning(
        env,
        Q,
        N_EP_TRAIN,
        learning_rate=fixed_alpha,
        gamma=g,
        min_epsilon=min_epsilon,
        max_epsilon=max_epsilon,
        decay_rate=decay_rate,
        rng=rng_q,
    )
    env.close()
    curves_gamma[g] = rh

plt.figure(figsize=(10, 5))
for g in gammas:
    y = smooth_curve(curves_gamma[g], SMOOTH_WIN)
    plt.plot(y, label=f"γ={g}")
plt.xlabel("Episódio (após suavização)")
plt.ylabel("Recompensa média (janela móvel)")
plt.title(f"Q-Learning: variação de γ (α={fixed_alpha} fixo)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Discussão (Tarefa 3 — preenchido com base nos gráficos)

- **Convergência mais rápida:** Indique qual curva sobe mais cedo (α / γ).
- **Melhor desempenho final:** Compare o patamar da recompensa suavizada no final do treino.

*Observação:* No Blackjack episódico sem desconto, **γ = 1** é o mais coerente com somar recompensa terminal; valores menores podem atrasar ou distorcer o valor das jogadas longas.

### Escolha dos melhores hiperparâmetros (para a Tarefa 4)

Ajuste as variáveis abaixo conforme os resultados dos gráficos acima.

In [ ]:
BEST_ALPHA = 0.01
BEST_GAMMA = 0.95
N_EP_COMPARE = N_EP_TRAIN
N_EVAL = 10_000

compare_rng = np.random.default_rng(SEED + 1)

---
## Tarefa 4 — Comparação Q-Learning vs SARSA vs Expected SARSA (1,5 pt)

Treino com os melhores α e γ da Tarefa 3; **mesmos** `decay_rate`, ε_min, ε_max e número de episódios. **Avaliação** greedy (sem exploração) em muitos episódios para média e desvio padrão.

In [ ]:
env_tr = gym.make(ENV_ID)

Q_ql = new_q_table()
Q_ql, rew_ql = train_q_learning(
    env_tr,
    Q_ql,
    N_EP_COMPARE,
    BEST_ALPHA,
    BEST_GAMMA,
    min_epsilon,
    max_epsilon,
    decay_rate,
    compare_rng,
)

Q_sa = new_q_table()
Q_sa, rew_sa = train_sarsa(
    env_tr,
    Q_sa,
    N_EP_COMPARE,
    BEST_ALPHA,
    BEST_GAMMA,
    min_epsilon,
    max_epsilon,
    decay_rate,
    compare_rng,
)

Q_es = new_q_table()
Q_es, rew_es = train_expected_sarsa(
    env_tr,
    Q_es,
    N_EP_COMPARE,
    BEST_ALPHA,
    BEST_GAMMA,
    min_epsilon,
    max_epsilon,
    decay_rate,
    compare_rng,
)

env_tr.close()

plt.figure(figsize=(11, 5))
for label, rew in [
    ("Q-Learning", rew_ql),
    ("SARSA", rew_sa),
    ("Expected SARSA", rew_es),
]:
    plt.plot(smooth_curve(rew, SMOOTH_WIN), label=label)
plt.xlabel("Episódio")
plt.ylabel("Recompensa (média móvel)")
plt.title("Comparação das curvas de aprendizado")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

env_ev = gym.make(ENV_ID)
rows = []
for name, Q in [
    ("Q-Learning", Q_ql),
    ("SARSA", Q_sa),
    ("Expected SARSA", Q_es),
]:
    m, s = evaluate_greedy(env_ev, Q, N_EVAL, seed_offset=999_000)
    rows.append((name, m, s))
env_ev.close()

print(f"{'Algoritmo':<18} {'Recompensa média':>18} {'Desvio padrão':>15}")
print("-" * 55)
for name, m, s in rows:
    print(f"{name:<18} {m:18.4f} {s:15.4f}")

### Análise escrita (Tarefa 4)

#### Q-Learning

O Q-Learning é *off-policy* e atualiza em direção ao máximo sobre as ações no estado seguinte. Na avaliação greedy obtive recompensa média **-0,0563** (desvio padrão ≈ **0,9537**). Esse desvio alto é esperado no Blackjack: em cada episódio o resultado é vitória, empate ou derrota, o que gera forte dispersão mesmo com política fixa. O valor médio ligeiramente negativo indica desempenho um pouco abaixo do neutro, típico de um agente forte mas não ótimo absoluto. O comportamento condiz com a ideia de aproximar a política greedy associada à função Q aprendida.

#### SARSA

O SARSA é *on-policy* e incorpora a exploração na transição seguinte. A média **-0,0544** (DP **0,9532**) é praticamente igual à do Q-Learning nesta execução (diferença da ordem de 0,002, irrelevante frente à variância). Isso é plausível no Blackjack, onde o risco de ações muito piores por exploração é mais limitado do que em grelhas com penhascos, por isso SARSA e Q-Learning tendem a ficar próximos. O desvio padrão semelhante ao dos outros algoritmos reflete a mesma variância intrínseca do ambiente.

#### Expected SARSA

O Expected SARSA segue a política ε-greedy mas usa o valor esperado da próxima ação em vez de uma única amostra, o que em teoria reduz a variância das atualizações em relação ao SARSA clássico. Na avaliação registou a melhor média entre os três: **-0,0457** (DP **0,9563**), ou seja, cerca de 0,01 melhor que Q-Learning e SARSA — ganho modesto mas consistente com updates mais estáveis durante o treino. O desvio permanece alto e próximo dos outros, como esperado neste ambiente.

#### Síntese

Os três métodos aprenderam (as curvas sobem a partir de valores próximos de -0,40). As diferenças entre Q-Learning e SARSA são mínimas nos teus resultados, o que condiz com a teoria para este domínio. O Expected SARSA apresentou o melhor desempenho médio na avaliação, alinhado à ideia de menor variância nas atualizações; ainda assim, as curvas de treino permanecem parecidas e os desvios são quase iguais, pelo que convém não exagerar a superioridade — pode reflectir ruído da seed e da estocástica dos episódios.
